# 📚 BDA_Project2_12 — MLii Ebook Fund Q&A Assistant
### Retrieval-Augmented Generation (RAG) Chatbot — Google Colab Edition

**Course:** Business Data Analytics — Project Part 2 (15 %)
**Group No.:** 12 (`BDA_Project2_12`)

| Student ID | Name |
|---|---|
| 6631501056 | TREEMONRAPAT VICHAISRI |
| 6631501055 | THEERATANA KOTIPANG |
| 6631501051 | THANYATHEP VITAYAKOVIT |
| 6631501063 | NAWAPHON PHROMPAO |
| 6631501086 | PHEERATHAD PANGPUTHIPONG |

---

### How to use this notebook
1. **Runtime → Run all**
2. When prompted, upload the dataset ZIP file.
3. Paste your Google API key when prompted.
4. The last cell launches the Streamlit app via localtunnel.

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
%%capture
!apt-get install -y tesseract-ocr tesseract-ocr-tha
!pip install -U pydantic protobuf google-api-core
!pip install -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface
!pip install -q sentence-transformers faiss-cpu pypdf python-docx
!pip install -q pytesseract Pillow streamlit pymupdf
print('✅ All libraries installed')

In [ ]:
# ============================================================
# Cell 2: Upload Dataset ZIP
# ============================================================
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

DATA_DIR = Path('/content/data')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for fname in uploaded:
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname) as zf:
            zf.extractall(DATA_DIR)
        print(f'✅ Unzipped {fname}')
    else:
        shutil.move(fname, DATA_DIR / fname)

print('\n📁 Files found:')
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_file():
        print(f'  - {p.relative_to(DATA_DIR)} ({p.stat().st_size//1024} KB)')

In [ ]:
# ============================================================
# Cell 3: Set Google API Key
# ============================================================
import os, getpass
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('🔑 API key loaded from Colab Secrets')
except Exception:
    pass
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Paste your Google API key: ')
    print('🔑 API key set for this session')

In [ ]:
# ============================================================
# Cell 4: Load Documents (DOCX / PDF with OCR Fallback / PNG)
# ============================================================
from pathlib import Path
from docx import Document as DocxDocument
from pypdf import PdfReader
from PIL import Image
import pytesseract, fitz
from langchain_core.documents import Document

DATA_DIR = Path('/content/data')

def load_docx(path):
    doc, parts = DocxDocument(path), []
    for p in doc.paragraphs:
        if p.text.strip(): parts.append(p.text.strip())
    for t in doc.tables:
        for row in t.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells: parts.append(' | '.join(cells))
    return '\n'.join(parts)

def load_pdf(path):
    try:
        text = '\n'.join((p.extract_text() or '') for p in PdfReader(str(path)).pages)
        if text.strip():
            return text
        # Scanned PDF — use OCR via PyMuPDF
        doc = fitz.open(str(path))
        parts = []
        for page in doc:
            pix = page.get_pixmap(dpi=200)
            img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
            parts.append(pytesseract.image_to_string(img, lang='tha+eng'))
        doc.close()
        return '\n'.join(parts).strip()
    except Exception as e:
        print(f'Warning: {path.name} — {e}')
        return ''

def load_png_ocr(path):
    try:
        return pytesseract.image_to_string(Image.open(path), lang='tha+eng').strip()
    except Exception as e:
        print(f'Warning: {path.name} — {e}')
        return ''

raw_docs = []
for path in sorted(DATA_DIR.rglob('*')):
    if not path.is_file(): continue
    s = path.suffix.lower()
    text = ''
    if   s == '.docx': text = load_docx(path)
    elif s == '.pdf':  text = load_pdf(path)
    elif s == '.png':  text = load_png_ocr(path)
    elif s == '.txt':  text = path.read_text(encoding='utf-8', errors='ignore')
    if text.strip():
        raw_docs.append(Document(page_content=text, metadata={'source': path.name}))

print(f'✅ Loaded {len(raw_docs)} documents')
for d in raw_docs:
    print(f'   - {d.metadata["source"]}: {len(d.page_content):,} chars')

In [ ]:
# ============================================================
# Cell 5: Chunk → Embed → Build FAISS Vectorstore
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=120,
    separators=['\n\n', '\n', ' ', '']
)
chunks = splitter.split_documents(raw_docs)
print(f'📄 {len(chunks)} chunks ready')

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
print('✅ FAISS vectorstore built')

In [ ]:
# ============================================================
# Cell 6: Build RAG Chain (Google Gemini)
# ============================================================
import os, json, subprocess

api_key = os.environ.get('GOOGLE_API_KEY')

def auto_detect_and_ask(query):
    try:
        docs = vectorstore.similarity_search(query, k=4)
        context = '\n\n'.join([d.page_content for d in docs])
        prompt = f'Context: {context}\n\nQuestion: {query}\n\nAnswer in Thai or English based on the question language:'

        list_url = f'https://generativelanguage.googleapis.com/v1beta/models?key={api_key}'
        list_proc = subprocess.Popen(['curl', '-s', list_url], stdout=subprocess.PIPE)
        list_res = json.loads(list_proc.communicate()[0].decode())

        model = None
        if 'models' in list_res:
            for m in list_res['models']:
                if 'generateContent' in m.get('supportedGenerationMethods', []):
                    model = m['name']
                    break
        if not model:
            return {'answer': 'No model found. Check API key.', 'source_documents': []}

        url = f'https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}'
        data = {'contents': [{'parts': [{'text': prompt}]}]}
        proc = subprocess.Popen(
            ['curl', '-s', '-X', 'POST', url, '-H', 'Content-Type: application/json', '-d', json.dumps(data)],
            stdout=subprocess.PIPE
        )
        res = json.loads(proc.communicate()[0].decode())

        if 'candidates' in res:
            answer = res['candidates'][0]['content']['parts'][0]['text']
        else:
            err = res.get('error', {}).get('message', 'Unknown error')
            answer = f'Error: {err}'

        return {'answer': answer, 'source_documents': docs}
    except Exception as e:
        return {'answer': f'Error: {str(e)}', 'source_documents': []}

chain = auto_detect_and_ask
print('✅ RAG chain ready')

In [ ]:
# ============================================================
# Cell 7: Quick Sanity Check (CLI Test)
# ============================================================
def ask(q):
    out = chain(q)
    print(f'Question: {q}')
    print(f'Answer:   {out["answer"]}')
    print('-' * 70)

ask('รายละเอียดการขอรับทุนตำรา 2 ประเภท แตกต่างกันอย่างไร')
ask('Who pays the reviewer fee for the type-1 grant?')
ask('ขั้นตอนการขอทุนประเภทที่ 1 มีกี่ขั้นตอน')

In [ ]:
# ============================================================
# Cell 8: Deploy Streamlit App
# ============================================================
import shutil, urllib.request

# 1. Copy app.py from GitHub repo (already correct standalone version)
#    If running locally from the zip, app.py is already in /content/
#    Otherwise download from your GitHub repo URL below:
# !wget -q https://raw.githubusercontent.com/<YOUR_GITHUB>/BDA_Project2_12/main/app.py

# 2. Write requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('langchain\nlangchain-community\nlangchain-core\nlangchain-text-splitters\n')
    f.write('langchain-huggingface\nsentence-transformers\nfaiss-cpu\n')
    f.write('pypdf\npython-docx\nstreamlit\npytesseract\nPillow\npymupdf\n')
print('✅ requirements.txt written')

# 3. Write packages.txt
with open('packages.txt', 'w') as f:
    f.write('tesseract-ocr\ntesseract-ocr-tha\n')
print('✅ packages.txt written')

# 4. Launch Streamlit via localtunnel
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print(f'\nPassword (your IP): {ip}')
print('Launching Streamlit...')
!streamlit run app.py &>/dev/null &
!npx localtunnel --port 8501

## Submission Checklist

| # | Item | Status |
|---|------|--------|
| 3.1 | Project name defined | ✅ BDA_Project2_12 |
| 3.2 | RAG pipeline built (Vibe code) | ✅ Cells 4–6 |
| 3.3 | Streamlit deployed | ✅ Cell 8 |
| 3.4 | Group No. + member list in UI | ✅ app.py |
| 3.5 | Video clip ≤ 5 mins | 🎥 Record after deployment |

**Files to submit:**
- `BDA_Project2_12.ipynb` — this notebook
- Public Streamlit URL
- `BDA_Project2_12.mp4` — presentation video